# JavaScript

All 18 JavaScript examples from [docs/extensions/javascript.md](https://platob.github.io/yggdryl/extensions/javascript/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

In [ ]:
const { DataType, Field, Url } = require('yggdryl')
const assert = require('node:assert/strict')

const schema = new Field(
  'row',
  DataType.fromFields([new Field('id', 'int64', false)]),
  false,
)

assert.equal(schema.dataType.kind, 'struct')
assert.equal(String(Url.fromPath('C:/market data/trades.arrows')),
  'file:///C:/market%20data/trades.arrows')

## Inference at the boundary

In [ ]:
const { DataType, Field, MediaType, MimeType, Url } = require('yggdryl')
const assert = require('node:assert/strict')

// A datatype expression is a datatype.
assert.equal(String(new Field('id', 'int64', false).dataType), 'int64')
assert.equal(DataType.from('list<int32>').kind, 'list')

// A media type is its canonical name.
assert.equal(String(MimeType.from('application/json')), 'application/json')
assert.equal(String(MediaType.from('application/json')), 'application/json')

// A path is a location.
assert.equal(String(Url.fromPath('C:/tmp/a.json')), 'file:///C:/tmp/a.json')

## Values cross as their natural shape

In [ ]:
const { json } = require('yggdryl')
const assert = require('node:assert/strict')

const decoded = json.loads(json.dumps({
  venues: new Set(['XPAR', 'XNAS']),
  book: new Map([[1, 'bid']]),
  source: new URL('https://example.com/feed'),
  match: /a\/b/giu,
  raw: Buffer.from([0, 255]),
  id: 2n ** 100n,
}))

assert.deepEqual(decoded.venues, ['XPAR', 'XNAS'])        // a Set is a list
assert.ok(decoded.book instanceof Map)                    // a non-text key keeps the Map
assert.equal(decoded.source, 'https://example.com/feed')  // a URL is its href
assert.equal(decoded.match, '/a\\/b/giu')                 // a RegExp is its literal
assert.deepEqual(decoded.raw, Buffer.from([0, 255]))
assert.equal(decoded.id, 2n ** 100n)

In [ ]:
const { json } = require('yggdryl')
const assert = require('node:assert/strict')

class Order {
  constructor(id) {
    this.id = id
  }
}

const decoded = json.loads(json.dumps({ order: new Order(7), venues: new Set(['XPAR']) }))
assert.deepEqual(decoded, { order: { id: 7 }, venues: ['XPAR'] })

const order = Object.assign(new Order(0), decoded.order)
assert.ok(order instanceof Order)
assert.deepEqual(new Set(decoded.venues), new Set(['XPAR']))

## Temporal and exact-decimal values

In [ ]:
const { Value, json } = require('yggdryl')
const assert = require('node:assert/strict')

const at = new Date('2026-08-15T12:30:00.000Z')
assert.ok(Value.fromJs(at).equals(Value.timestamp(1786797000000n, 'ms')))
assert.ok(json.loads(json.dumps(at)) instanceof Date)

// A microsecond instant in a named zone is not a Date, so it stays exact.
const micros = Value.timestamp(1700000000123456n, 'us', 'UTC')
const decoded = json.loads(json.dumps({ at: micros })).at
assert.equal(decoded.kind, 'timestamp')
assert.equal(decoded.count, 1700000000123456n)
assert.equal(decoded.unit, 'us')
assert.equal(decoded.zone, 'UTC')
assert.ok(decoded.equals(micros))

In [ ]:
const { Value } = require('yggdryl')
const assert = require('node:assert/strict')

const price = Value.decimal(-1050n, 2) // -10.50
assert.equal(price.unscaled, -1050n)
assert.equal(price.scale, 2)

// Equality is what a value names, not how it was written.
assert.ok(price.equals(Value.decimal(-105n, 1)))
assert.ok(Value.duration(1n, 's').equals(Value.duration(1000n, 'ms')))
assert.equal(Value.date(19723).unit, null)

## fromJs and asJs

In [ ]:
const { Value, json } = require('yggdryl')
const assert = require('node:assert/strict')

assert.equal(Value.fromJs(new Set([1, 2])).kind, 'sequence')
assert.deepEqual(Value.fromJs(new Set([1, 2])).asJs(), [1, 2])
assert.equal(Value.fromJs(new Map([['id', 1]])).kind, 'mapping')

const value = { id: 1, at: new Date(0), tags: new Set(['a']) }
assert.deepEqual(json.loads(json.dumps(value)), Value.fromJs(value).asJs())

## Field metadata is a Map

In [ ]:
const { Field } = require('yggdryl')
const assert = require('node:assert/strict')

const field = new Field('trade', 'int64', false, { source: 'book' })
field.set('venue', 'XPAR')

assert.equal(field.get('source'), 'book')
assert.ok(field.has('venue'))
assert.equal(field.size, 2)
assert.deepEqual([...field.keys()].sort(), ['source', 'venue'])

field.delete('venue')
assert.ok(!field.has('venue'))

In [ ]:
const { Field } = require('yggdryl')
const assert = require('node:assert/strict')

const field = new Field('price', 'int64', false)
field.iceberg.set('doc', 'closing price')
field.postgres.update({ type: 'numeric' })

assert.equal(field.iceberg.get('doc'), 'closing price')
assert.deepEqual([...field.postgres], [['type', 'numeric']])
assert.equal(field.iceberg.size, 1)
assert.equal(field.postgres.has('doc'), false)

// The bare name is all the view needs; the full key is what the field stores.
assert.equal(field.iceberg.key('doc'), 'iceberg:doc')
assert.equal(field.get('iceberg:doc'), 'closing price')
assert.equal(field.size, 2)

assert.equal(field.iceberg.delete('doc'), true)
assert.equal(field.iceberg.size, 0)

In [ ]:
const { DataType, Field } = require('yggdryl')
const assert = require('node:assert/strict')

const schema = new Field(
  'row',
  DataType.fromFields([
    new Field('year', 'int32', false),
    new Field('price', 'int64', false),
  ]),
  false,
).withPartitionFields(['year'])

assert.deepEqual(schema.partitionFieldNames(), ['year'])
assert.equal(schema.dataType.getByName('year').isPartition, true)
assert.equal(schema.withoutPartitionFields().dataType.length, 1)

## Arrow

In [ ]:
const { DataType } = require('yggdryl')
const assert = require('node:assert/strict')

const scalar = DataType.from('int64').defaultArrowScalar()
assert.equal(String(scalar), '0')

## Records cross as one batch per stream

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { BatchReader, IOBase, MimeType } = require('yggdryl')

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
})

// An in-memory handle says what it holds; a named one reads it off its name.
const handle = IOBase.fromBytes()
handle.mediaType = MimeType.ARROW_STREAM
handle.writeArrowBatchReader(BatchReader.from(table))

const reader = handle.readArrowBatchReader()
assert.equal(reader.field.name, 'row')
assert.equal([...reader].reduce((rows, batch) => rows + batch.numRows, 0), 2)

// A stream is read once, and says so rather than reading as empty.
assert.ok(reader.consumed)

In [ ]:
const assert = require('node:assert/strict')
const { RecordOptions } = require('yggdryl')

const parquet = RecordOptions.from('trades.parquet')
assert.equal(String(parquet.mimeType), 'application/vnd.apache.parquet')
assert.equal(parquet.compression, 'zstd(1)')
assert.equal(parquet.withCompression('snappy').compression, 'snappy')

// A setting one encoding has is absent on the others rather than invented.
const stream = RecordOptions.from('trades.arrows')
assert.equal(stream.compression, null)
assert.equal(stream.maxRowGroupSize, null)

## Anything in, a reader out

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { IOBase, MimeType } = require('yggdryl')

const table = new arrow.Table({
  id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
})

function handle() {
  const stream = IOBase.fromBytes()
  stream.mediaType = MimeType.ARROW_STREAM
  return stream
}

// A table, a reader, named columns, and plain records all write.
for (const rows of [
  table,
  arrow.RecordBatchReader.from(arrow.tableToIPC(table)),
  { id: [1n, 2n] },
  [{ id: 1n }, { id: 2n }],
]) {
  const target = handle()
  target.writeArrow(rows)
  assert.equal(target.readArrow().toTable().numRows, 2)
}

In [ ]:
const assert = require('node:assert/strict')
const arrow = require('apache-arrow')
const { IOBase, MimeType } = require('yggdryl')

async function main() {
  const table = new arrow.Table({ id: arrow.vectorFromArray([1n], new arrow.Int64()) })
  async function* pages() {
    yield table
    yield table
  }

  const handle = IOBase.fromBytes()
  handle.mediaType = MimeType.ARROW_STREAM
  await handle.writeArrow(pages())

  assert.equal(handle.readArrow().toTable().numRows, 2)
}

main()

## Iceberg is a namespace

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { Field, fields, iceberg } = require('yggdryl')

const schema = fields.struct('row', [Field.from('id: int64'), Field.from('venue: utf8')], {
  nullable: false,
})

const root = path.join(fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-docs-')), 'trades')
const table = iceberg.Table.create(root, schema, ['venue'])
table.append(
  new arrow.Table({
    id: arrow.vectorFromArray([1n, 2n], new arrow.Int64()),
    venue: arrow.vectorFromArray(['XNAS', 'XNYS'], new arrow.Utf8()),
  }),
)

assert.equal(table.currentSnapshot.operation, 'append')
assert.equal(table.dataFiles().length, 2)
assert.equal(table.scan().toTable().numRows, 2)

fs.rmSync(path.dirname(root), { recursive: true, force: true })

## An Iceberg table end to end

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const arrow = require('apache-arrow')
const { iceberg } = require('yggdryl')

const warehouse = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-doc-'))
const catalog = new iceberg.Catalog(warehouse)

// Rows and a dotted name are enough: the first append creates the table.
const rows = (ids, venues) =>
  new arrow.Table({
    id: arrow.vectorFromArray(ids, new arrow.Int64()),
    venue: arrow.vectorFromArray(venues, new arrow.Utf8()),
  })
const table = catalog.append('nyc.trades', rows([1n, 2n], ['XNAS', 'XNYS']))
const past = table.currentSnapshot.snapshotId
table.append(rows([3n], ['XASE']))
assert.deepEqual(catalog.listTables('nyc'), ['nyc.trades'])
assert.equal(table.scan().toTable().numRows, 3)

// A column change is a chain recorded on the update, committed once.
table.updateSchema().addColumn('', 'price: float64').commit()
assert.equal(table.scan().toTable().getChild('price').get(0), null)

// Undersized files rewrite as one replace commit that reports itself.
const compaction = table.compact()
assert.equal(compaction.filesBefore, 2)
assert.equal(compaction.filesAfter, 1)
assert.equal(table.scan().toTable().numRows, 3)

// And nothing rewrote history: the first snapshot reads as it was written.
assert.deepEqual(
  table.scanAt(past).toTable().getChild('id').toArray(),
  BigInt64Array.from([1n, 2n]),
)

fs.rmSync(warehouse, { recursive: true, force: true })

## Errors

In [ ]:
const { DataType } = require('yggdryl')
const assert = require('node:assert/strict')

assert.throws(() => DataType.from('decimal(0,0)'), /precision/)